$\newcommand{\ket}[1]{\left\lvert #1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right\rvert}
\newcommand{\braket}[1]{\left\langle #1\right\rangle}
\newcommand{\proj}[1]{\lvert #1\rangle\!\langle #1\rvert}
\newcommand{\tr}{\operatorname{tr}}\newcommand{\Tr}{\operatorname{Tr}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\rank}{\operatorname{rank}}\newcommand{\diag}{\operatorname{diag}}
\newcommand{\poly}{\operatorname{poly}}\newcommand{\polylog}{\operatorname{polylog}}
\newcommand{\norm}[1]{\lVert #1\rVert}\newcommand{\abs}[1]{\lvert #1\rvert}
\newcommand{\bigo}{\mathcal{O}}\newcommand{\id}{\mathbb{I}}\newcommand{\dg}{^{\dagger}}
\newcommand{\R}{\mathbb{R}}\newcommand{\C}{\mathbb{C}}\newcommand{\N}{\mathbb{N}}
\newcommand{\Z}{\mathbb{Z}}\newcommand{\E}{\mathbb{E}}\newcommand{\Prob}{\mathbb{P}}
\newcommand{\Hilb}{\mathcal{H}}\newcommand{\ii}{\mathrm{i}}\newcommand{\dd}{\mathrm{d}}
\newcommand{\ee}{\mathrm{e}}\newcommand{\QFT}{\mathrm{QFT}}\newcommand{\CNOT}{\mathrm{CNOT}}
\newcommand{\Uf}{U_f}\newcommand{\defeq}{:=}
\newcommand{\Xgate}{\mathsf{X}}\newcommand{\Ygate}{\mathsf{Y}}
\newcommand{\Zgate}{\mathsf{Z}}\newcommand{\Hgate}{\mathsf{H}}$

# Linear Combination of Unitaries and Block-Encoding Arithmetic

**Learning objectives.**

-   Implement a weighted sum of unitaries with the [prepare]{.smallcaps}--[select]{.smallcaps}--[prepare]{.smallcaps}$\dg$ (LCU) circuit and read off its block-encoding parameters.

-   Realize matrix and vector addition via LCU as a block encoding with subnormalization equal to the $\ell^1$-norm of the coefficients.

-   Compose block encodings to multiply matrices, and explain why naive stacking fails while duplicate-ancilla and compression-gadget constructions succeed.

-   Contrast the coherent LCU circuit with a hybrid importance-sampling estimator, and compare their costs.

-   Track the $\ell^1$ overhead $\norm{c}_1$ through additions and the product $\prod\alpha_j$ through multiplications, and see when amplitude amplification removes the associated success-probability penalty.

The previous lecture (that chapter) delivered two input models: the
state-preparation oracle for vectors and the block encoding for matrices. A
single block encoding is, however, only as useful as the arithmetic we can build
on top of it. Almost every matrix that arises in scientific computing is presented
as a *combination* of simpler pieces---a Hamiltonian is a sum of local terms,
a discretized differential operator is a sum of shifts, a propagator is a product
of exponentials---so we need rules that turn block encodings of the pieces into a
block encoding of the whole. This lecture develops the single most important such
rule, the *linear combination of unitaries* (LCU), and the composition
calculus it generates: block encodings of matrix and vector *sums*
(that section) and *products* (that section). LCU is the
[prepare]{.smallcaps}--[select]{.smallcaps}--[prepare]{.smallcaps}$\dg$ sandwich already
previewed in that example; here we derive it in full, quantify its success
probability and repeat count, and note how oblivious amplitude amplification
(that chapter) removes the subnormalization penalty when the target is
(nearly) an isometry. We then contrast the coherent circuit with a purely
classical *hybrid* alternative---importance sampling over the terms---whose
appeal and whose variance we analyze in that section. The recurring theme,
made precise in that section, is that the cost of LCU is governed by the
$\ell^1$-norm $\norm{c}_1=\sum_j\abs{c_j}$ of the combination coefficients, exactly
as the cost of a single block encoding was governed by its subnormalization factor
$\alpha$. Throughout, $A$ is an $n$-qubit matrix ($N=2^n$) and we reuse the
block-encoding notation $\mathrm{BE}_{\alpha,a}(A,\epsilon)$ of
that definition.

## The LCU construction

Suppose a target operator is presented as a weighted sum of $K$ unitaries,
$$T=\sum_{j=0}^{K-1}c_j\,U_j,\qquad U_j\in\mathrm{U}(2^n),\quad c_j>0,\quad K=2^a.$$
(We take the coefficients positive for now; a phase $c_j=\abs{c_j}\ee^{\ii\phi_j}$
is absorbed into $U_j\mapsto\ee^{\ii\phi_j}U_j$, still unitary, so the general real
or complex case reduces to this one.) The idea of LCU is to encode the coefficients
in the amplitudes of an $a$-qubit *control* register, apply the $U_j$ in
superposition controlled on that register, and then coherently "fold" the
superposition back so that the flag $\ket{0^a}$ picks out precisely $T$. The two
ingredients are the [select]{.smallcaps} and [prepare]{.smallcaps} oracles.

**Definition ([select]{.smallcaps} and [prepare]{.smallcaps} oracles).**

The *select oracle* for the family $\{U_j\}_{j\in[K]}$ is the $(a+n)$-qubit
unitary
$$\mathrm{SEL}\defeq\sum_{j\in[K]}\proj{j}\otimes U_j,$$
which applies $U_j$ to the system conditioned on the control register holding
$\ket{j}$. The *prepare oracle* for the coefficients $c=(c_0,\dots,c_{K-1})$,
$c_j\ge0$, is any $a$-qubit unitary $\mathrm{PREP}$ with
$$\mathrm{PREP}\ket{0^a}=\frac{1}{\sqrt{\norm{c}_1}}\sum_{j\in[K]}\sqrt{c_j}\,\ket{j},
  \qquad \norm{c}_1=\sum_{j\in[K]}\abs{c_j}.$$

Only the first column of $\mathrm{PREP}$ is fixed by the equation above; the
remaining columns may be any completion to a unitary (its existence is guaranteed
because the specified column is a unit vector). The following lemma is the
workhorse of Part II.

**Theorem (Linear combination of unitaries).**

Let $T=\sum_{j\in[K]}c_j U_j$ with $c_j\ge0$, $K=2^a$, and let
$\mathrm{SEL},\mathrm{PREP}$ be as in that definition. Then
$$W\defeq\bigl(\mathrm{PREP}\dg\otimes\id_n\bigr)\,\mathrm{SEL}\,
          \bigl(\mathrm{PREP}\otimes\id_n\bigr)$$
is an exact $(\norm{c}_1,a)$-block-encoding of $T$, i.e. $W\in\mathrm{BE}_{\norm{c}_1,a}(T)$.

*Proof.*

Fix an arbitrary $n$-qubit state $\ket{\psi}$ and propagate the flagged input
$\ket{0^a}\ket{\psi}$ through the three factors. First $\mathrm{PREP}$ dresses the
control register with the coefficient amplitudes,
$$\bigl(\mathrm{PREP}\otimes\id\bigr)\ket{0^a}\ket{\psi}
  =\frac{1}{\sqrt{\norm{c}_1}}\sum_{j}\sqrt{c_j}\,\ket{j}\ket{\psi}.$$
Next $\mathrm{SEL}$ applies $U_j$ in each branch:
$$\mathrm{SEL}\,\frac{1}{\sqrt{\norm{c}_1}}\sum_{j}\sqrt{c_j}\,\ket{j}\ket{\psi}
  =\frac{1}{\sqrt{\norm{c}_1}}\sum_{j}\sqrt{c_j}\,\ket{j}\,U_j\ket{\psi}.$$
Finally $\mathrm{PREP}\dg$ acts on the control register. We do not need the full
image; we need only the amplitude of the flag $\ket{0^a}$, obtained by projecting
with $\bra{0^a}$. Using
$\bra{0^a}\mathrm{PREP}\dg=\bigl(\mathrm{PREP}\ket{0^a}\bigr)\dg
=\frac{1}{\sqrt{\norm{c}_1}}\sum_{j}\sqrt{c_j}\,\bra{j}$ and
$\braket{j'|j}=\delta_{jj'}$,
$$\bigl(\bra{0^a}\otimes\id\bigr)W\bigl(\ket{0^a}\otimes\id\bigr)\ket{\psi}
  =\frac{1}{\norm{c}_1}\sum_{j,j'}\sqrt{c_{j'}}\sqrt{c_j}\,
   \braket{j'|j}\,U_j\ket{\psi}
  =\frac{1}{\norm{c}_1}\sum_{j}c_j U_j\ket{\psi}
  =\frac{T}{\norm{c}_1}\ket{\psi}.$$
Since $\ket{\psi}$ was arbitrary,
$(\bra{0^a}\otimes\id)W(\ket{0^a}\otimes\id)=T/\norm{c}_1$, which is exactly the
exact-block-encoding relation the equation above with subnormalization factor
$\alpha=\norm{c}_1$ and $a$ ancilla qubits.

Three remarks fix the meaning of the construction.

**Remark (Reading the circuit; why $\ell^1$).**

The output of $W$ on $\ket{0^a}\ket{\psi}$ is, keeping all branches,
$$W\ket{0^a}\ket{\psi}
  =\frac{1}{\norm{c}_1}\ket{0^a}\,T\ket{\psi}+\ket{\perp},
  \qquad \bigl(\proj{0^a}\otimes\id\bigr)\ket{\perp}=0,$$
so the flag $\ket{0^a}$ is measured with probability
$\norm{T\ket{\psi}}^2/\norm{c}_1^2$, and conditioned on it the system holds the
normalized $T\ket{\psi}/\norm{T\ket{\psi}}$, exactly as in the general
block-encoding picture (that section). The subnormalization is the
$\ell^1$-norm $\norm{c}_1$, *not* $\norm{T}$: because
$\norm{T}\le\sum_j c_j\norm{U_j}=\norm{c}_1$, LCU is never better than
$\alpha=\norm{c}_1$, and it is tight exactly when the $U_j$ add up "in phase"
(e.g. all $U_j=\id$). Reducing $\norm{c}_1$---by choosing a more efficient
decomposition of $T$ into unitaries---is the central knob for controlling cost, and
is the subject of that section.

**Remark (Cost is logarithmic in the number of terms).**

The register that carries the combination has only $a=\log_2 K$ qubits, so LCU can
superpose an exponentially large number of terms with a linear number of ancillas.
The genuine cost hides in the two oracles: $\mathrm{PREP}$ must load the
$\sqrt{c_j}$ amplitudes (that definition, subject to the same structure
requirements as any state preparation) and $\mathrm{SEL}$ must apply a
$K$-fold-multiplexed unitary. A direct implementation of either scales linearly in
$K$; an *efficient* LCU therefore needs additional structure---for instance,
the $U_j$ being Pauli strings addressed by their binary label, so that $\mathrm{SEL}$
is a sequence of $a$ controlled-Pauli layers.

**Remark (Unequal split of the coefficients).**

Nothing forces the "load" and "unload" oracles to be adjoints. Writing
$c_j=\beta_j\gamma_j$ and using two preparations
$\mathrm{PREP}\ket{0^a}=\frac{1}{\norm{\beta}_2}\sum_j\beta_j\ket{j}$ and
$\widetilde{\mathrm{PREP}}\ket{0^a}=\frac{1}{\norm{\gamma}_2}\sum_j\gamma_j\ket{j}$,
the sandwich $W=(\widetilde{\mathrm{PREP}}\dg\otimes\id)\,\mathrm{SEL}\,
(\mathrm{PREP}\otimes\id)$ is a $(\norm{\beta}_2\norm{\gamma}_2,a)$-block-encoding of
$T$. By Cauchy--Schwarz $\norm{c}_1=\sum_j\abs{\beta_j\gamma_j}\le
\norm{\beta}_2\norm{\gamma}_2$, so the symmetric choice
$\beta_j=\gamma_j=\sqrt{c_j}$ (whence
$\widetilde{\mathrm{PREP}}=\mathrm{PREP}$) minimizes the subnormalization at
$\norm{c}_1$. The freedom is occasionally useful when one of the two preparations is
cheaper to build than its adjoint.

**Example (LCU of two Pauli matrices).**

Take the single-qubit Hermitian operator $T=\tfrac12\Xgate+\tfrac12\Zgate$, so
$K=2$, $a=1$, $c_0=c_1=\tfrac12$, $U_0=\Xgate$, $U_1=\Zgate$, and
$\norm{c}_1=1$. The prepare oracle must satisfy
$\mathrm{PREP}\ket{0}=\tfrac{1}{\sqrt1}\bigl(\sqrt{\tfrac12}\ket{0}
+\sqrt{\tfrac12}\ket{1}\bigr)=\tfrac{1}{\sqrt2}(\ket0+\ket1)$, which is the
Hadamard gate $\mathrm{PREP}=\Hgate$ (self-inverse, so
$\mathrm{PREP}\dg=\Hgate$). The select oracle is
$\mathrm{SEL}=\proj{0}\otimes\Xgate+\proj{1}\otimes\Zgate$, a controlled-$\Xgate$
followed by a control-on-$\ket1$ $\Zgate$. Then
$W=(\Hgate\otimes\id)\,\mathrm{SEL}\,(\Hgate\otimes\id)$ and, by
the equation above,
$$(\bra0\otimes\id)W(\ket0\otimes\id)
  =\tfrac12\Xgate+\tfrac12\Zgate
  =\begin{pmatrix}\tfrac12&\tfrac12\\[2pt]\tfrac12&-\tfrac12\end{pmatrix}=T,$$
so $W\in\mathrm{BE}_{1,1}(T)$. Here $\alpha=\norm{c}_1=1$ is optimal: the two
Paulis are orthogonal in Hilbert--Schmidt inner product, $\norm{T}=\tfrac{1}{\sqrt2}
\cdot\sqrt2=1$ (its singular values are both $1/\sqrt2\cdot\sqrt2$; explicitly
$T/\!\sqrt2$ is a reflection, $\norm{T}=1$), so no block encoding can do better than
$\alpha=1$. Applying $W$ to $\ket0\ket\psi$ succeeds with probability
$\norm{T\ket\psi}^2$, which equals $1$ for every $\ket\psi$ because $T$ is (up to the
scalar) unitary; LCU here needs no repetition.

**Remark (Oblivious amplitude amplification).**

When $T$ is itself unitary (or an isometry), as in that example with
$\norm{c}_1=1$, the flag $\ket{0^a}$ still appears only with probability
$\norm{T\ket\psi}^2/\norm{c}_1^2$, which is $<1$ whenever $\norm{c}_1>1$. Naively one
would amplify the flag with amplitude amplification (that chapter), but the
standard routine needs to reflect about the *input* state $\ket{\psi}$, which
is unknown and possibly entangled. *Oblivious* amplitude amplification removes
this obstacle: because $W$ acts as the fixed $2\times2$ rotation
$\bigl(\begin{smallmatrix}\sin\theta&\cos\theta\\ \cos\theta&-\sin\theta
\end{smallmatrix}\bigr)$ (with $\sin\theta=\norm{T}/\norm{c}_1$ when $T$ is
proportional to an isometry) on the two-dimensional subspace spanned by
$\ket{0^a}\ket\psi$ and $\ket\perp$ *for every* $\ket\psi$ simultaneously, one
can reflect about $\ket{0^a}$ on the ancilla *alone* and drive the flag to
certainty in $\bigo(\norm{c}_1/\norm{T})$ rounds, independently of $\ket\psi$. This
is why a Hamiltonian written as $H=\sum_j c_j P_j$ with the $P_j$ Pauli strings can
be applied deterministically up to the $\ell^1$ overhead. We use this repeatedly in
the Taylor-series simulation method of that chapter.

## Matrix and vector addition

The LCU lemma applies verbatim when the "unitaries" $U_j$ are themselves block
encodings or state-preparation oracles: the outer register of the block encoding is
merged into the LCU flag, and the surviving block is the desired sum. This is the
first arithmetic rule of the block-encoding calculus.

**Proposition (Sum of block-encoded matrices).**

Let $M=2^m$ and let $A_0,\dots,A_{M-1}$ be $n$-qubit matrices, each supplied by a
block encoding $U_{A_j}\in\mathrm{BE}_{\alpha_j,a}(A_j,\epsilon_j)$ with
$\alpha_j\ge0$. Set $\gamma\defeq\sum_{j\in[M]}\alpha_j$. Let
$\mathrm{SEL}=\sum_{j\in[M]}\proj{j}\otimes U_{A_j}$ act on an $m$-qubit control
register together with the shared $a$-qubit flag and system, and let $\mathrm{PREP}$
be any $m$-qubit unitary with
$\mathrm{PREP}\ket{0^m}=\frac{1}{\sqrt\gamma}\sum_{j\in[M]}\sqrt{\alpha_j}\,\ket{j}$.
Then
$$W\defeq\bigl(\mathrm{PREP}\dg\otimes\id\bigr)\,\mathrm{SEL}\,
          \bigl(\mathrm{PREP}\otimes\id\bigr)
  \in\mathrm{BE}_{\gamma,\,a+m}\!\Bigl(\textstyle\sum_{j\in[M]}A_j,\;
     \sum_{j\in[M]}\epsilon_j\Bigr).$$
More generally, for a weighted sum $\sum_j c_j A_j$ with $c_j\ge0$ one rescales the
prepared amplitudes to $\sqrt{c_j\alpha_j}$ and obtains subnormalization
$\sum_j c_j\alpha_j$.

*Proof.*

Write $B_j\defeq(\bra{0^a}\otimes\id)U_{A_j}(\ket{0^a}\otimes\id)$, the block of
$U_{A_j}$, so that $\norm{A_j-\alpha_j B_j}\le\epsilon_j$ and (a block of a unitary)
$\norm{B_j}\le1$. The flag for $W$ is the joint state $\ket{0^m}\ket{0^a}$. Running
the three factors exactly as in that theorem, but now with each branch's
system operator being the block $B_j$ rather than a full unitary, gives
$$\bigl(\bra{0^m}\bra{0^a}\otimes\id\bigr)W
  \bigl(\ket{0^m}\ket{0^a}\otimes\id\bigr)
  =\frac{1}{\gamma}\sum_{j\in[M]}\alpha_j B_j.$$
The only new point is that $\mathrm{SEL}$ leaves the $a$-qubit flag of $U_{A_j}$
where it is, so the $\ket{0^m}\ket{0^a}$ projection extracts $\alpha_j B_j/\gamma$
in each branch; the $\ket{j}$-register bookkeeping is identical to
the equation above. Comparing with the target $\gamma^{-1}\sum_j A_j$ and using
the triangle inequality,
$$\Bigl\lVert\,\sum_{j}A_j-\gamma\cdot\tfrac1\gamma\sum_j\alpha_j B_j\Bigr\rVert
  =\Bigl\lVert\sum_j(A_j-\alpha_j B_j)\Bigr\rVert
  \le\sum_j\norm{A_j-\alpha_j B_j}\le\sum_j\epsilon_j,$$
which is exactly the block-encoding statement the equation above. The weighted
case follows by replacing $\alpha_j$ with $c_j\alpha_j$ in $\mathrm{PREP}$.

The subnormalization of a sum is the sum of the subnormalizations, and the errors
add. This linear growth is benign for a fixed number of terms but is precisely the
$\ell^1$ overhead that we must budget for; see that section.

**Example (Block-encoding the average $(A+B)/2$).**

Let $U_A\in\mathrm{BE}_{1,a}(A)$ and $U_B\in\mathrm{BE}_{1,a}(B)$ be two exact
block encodings with the same flag register ($\norm{A},\norm{B}\le1$). Take
$M=2$, $m=1$, weights $c_0=c_1=\tfrac12$, so $\gamma=\sum_j c_j\alpha_j=\tfrac12+
\tfrac12=1$ and the prepared state is
$\mathrm{PREP}\ket0=\tfrac{1}{\sqrt2}(\ket0+\ket1)$, i.e. $\mathrm{PREP}=\Hgate$.
With $\mathrm{SEL}=\proj0\otimes U_A+\proj1\otimes U_B$, the circuit
$W=(\Hgate\otimes\id)\,\mathrm{SEL}\,(\Hgate\otimes\id)$ satisfies
$$(\bra0\bra{0^a}\otimes\id)W(\ket0\ket{0^a}\otimes\id)
  =\tfrac12 A+\tfrac12 B=\frac{A+B}{2},$$
so $W\in\mathrm{BE}_{1,\,a+1}\!\bigl(\tfrac{A+B}{2}\bigr)$. The single control qubit
selects between the two block encodings; the average incurs *no*
subnormalization penalty ($\gamma=1$) because the weights already sum to one. Had we
instead block-encoded $A+B$ (weights $1,1$) the subnormalization would be
$\gamma=2$, reflecting that $\norm{A+B}$ can be as large as $2$.

### Vector addition {#vector-addition .unnumbered}

The same circuit adds *vectors* when the $U_j$ are state-preparation oracles
rather than block encodings. If $O_j\ket{0^n}=\ket{u_j}$ prepares the (normalized)
$j$-th vector (that definition) and we wish to prepare the superposition
$\sum_j c_j\ket{u_j}$ (with $c_j\ge0$), set $\mathrm{SEL}=\sum_j\proj{j}\otimes O_j$
and $\mathrm{PREP}\ket{0^m}=\frac{1}{\sqrt{\norm{c}_1}}\sum_j\sqrt{c_j}\ket{j}$. Then
$\mathrm{SEL}(\mathrm{PREP}\otimes\id)\ket{0^m}\ket{0^n}
=\frac{1}{\sqrt{\norm{c}_1}}\sum_j\sqrt{c_j}\ket{j}\ket{u_j}$, and applying
$\mathrm{PREP}\dg$ and post-selecting the flag $\ket{0^m}$ yields
$$\frac{1}{\norm{c}_1}\ket{0^m}\sum_{j}c_j\ket{u_j}+\ket{\perp},$$
so the flag is seen with probability $\norm{\sum_j c_j\ket{u_j}}^2/\norm{c}_1^2$ and
the system then holds the (renormalized) target. This is the vector counterpart of
that result: the "matrix" $U_j$ is replaced by the "vector" oracle
$O_j$, and the surviving flag block is the vector sum. It is the primitive behind,
for example, forming $\sum_k \hat f(k)\,\ee^{\ii k H}\ket{\psi}$ in Fourier-based
matrix-function algorithms.

## Block-encoding multiplication

Addition composed block encodings side by side, sharing the system register. To
*multiply* $A$ and $B$ we must feed the output of one into the input of the
other. The subtlety is that the block-encoding relation only holds on the flagged
subspace, so we cannot simply apply $U_A$ and then $U_B$ on the same flag qubits.

**Remark (Why naive stacking fails).**

Suppose $U_A\in\mathrm{BE}_{\alpha,a}(A)$ and $U_B\in\mathrm{BE}_{\beta,a}(B)$ share
the *same* $a$-qubit flag. Applying them in sequence gives
$$U_B U_A\ket{0^a}\ket\psi
  =U_B\Bigl(\tfrac{1}{\alpha}\ket{0^a}A\ket\psi+\ket{\perp_A}\Bigr),$$
and the $\ket{\perp_A}$ branch---which is *not* flagged by $\ket{0^a}$---is
mapped by $U_B$ back into the flagged subspace in an uncontrolled way. Concretely
$(\bra{0^a}\otimes\id)U_BU_A(\ket{0^a}\otimes\id)$ picks up cross terms from
$\ket{\perp_A}$ and does *not* equal $\frac{1}{\alpha\beta}BA$. The remedy is to
keep the two flag registers *separate*, so that returning to the joint flag
$\ket{0^{a}}\ket{0^{b}}$ forces *both* block encodings to have acted on their
flagged subspaces.

**Proposition (Product of two block-encoded matrices).**

Let $U_A\in\mathrm{BE}_{\alpha,a}(A)$ and $U_B\in\mathrm{BE}_{\beta,b}(B)$ act on
disjoint ancilla registers (of sizes $a$ and $b$) and a common $n$-qubit system.
Then
$$U_{AB}\defeq\bigl(U_A\otimes\id_b\bigr)\bigl(\id_a\otimes U_B\bigr)
  \in\mathrm{BE}_{\alpha\beta,\,a+b}(AB),$$
using $a+b$ ancilla qubits. (Here $U_A$ acts on flag register $a$ and the system,
$U_B$ on flag register $b$ and the system, and the two flag registers do not
overlap.)

*Proof.*

The joint flag is $\ket{0^a}\ket{0^b}$. Because the two flag registers are disjoint,
$(\bra{0^a}\otimes\id)$ and $(\bra{0^b}\otimes\id)$ commute past each other and the
block of the product factorizes:
$$\begin{aligned}
  \bigl(\bra{0^a}\bra{0^b}\otimes\id\bigr)U_{AB}
  \bigl(\ket{0^a}\ket{0^b}\otimes\id\bigr)
  &=\bigl(\bra{0^a}\otimes\id\bigr)U_A\bigl(\ket{0^a}\otimes\id\bigr)\;
    \bigl(\bra{0^b}\otimes\id\bigr)U_B\bigl(\ket{0^b}\otimes\id\bigr)\notag\\
  &=\frac{A}{\alpha}\cdot\frac{B}{\beta}=\frac{AB}{\alpha\beta}.
  
\end{aligned}$$
The middle step is the key: $\id_a\otimes U_B$ leaves flag register $a$ untouched,
so projecting it onto $\ket{0^a}$ before and after commutes through $U_B$; likewise
$U_A\otimes\id_b$ leaves flag register $b$ untouched. Thus the two blocks separate
and multiply. Comparing with the exact-encoding relation
the equation above gives $U_{AB}\in\mathrm{BE}_{\alpha\beta,a+b}(AB)$. (With
errors $\epsilon_A,\epsilon_B$ the same telescoping bound as in
that result gives error $\alpha\beta$ times the product-rule expansion; the
exact case suffices for us.)

The product multiplies the subnormalizations, $\alpha\beta$, and *adds* the
ancilla counts, $a+b$. Iterating the equation above over $L$ factors gives
$\prod_{j}\alpha_j$ subnormalization and $\sum_j a_j$ ancillas---correct, but
ancilla-hungry.

**Remark (Duplicate-ancilla vs. compression-gadget cost).**

Two refinements reduce the ancilla count for a product of $L$ block encodings, each
in $\mathrm{BE}_{\alpha,a}(A_i)$.

-   *Duplicate ancilla.* A deferred-measurement trick records, in a single
    extra qubit, whether *all* flag registers returned to $\ket{0^a}$; one
    can then reuse the flag register across the $L$ factors and needs only
    $a+(L-1)$ ancillas (one bookkeeping qubit per factor beyond the first) to
    obtain a unitary in $\mathrm{BE}_{\alpha^{L},\,a+L-1}(A_L\cdots A_1)$.

-   *Compression gadget.* The bookkeeping qubits are highly redundant---one
    only needs to know whether *any* factor left the flag subspace, a single
    bit of information that a counter can hold. The compression gadget replaces
    the $L-1$ bookkeeping qubits by a $\lceil\log_2(L+1)\rceil$-qubit counter,
    giving $\mathrm{BE}_{\alpha^{L},\,a+\lceil\log_2(L+1)\rceil}(A_L\cdots A_1)$.
    This exponential saving in ancillas is what makes long products (e.g. Chebyshev/monomial expansions inside QSVT, that chapter) practical.

Both refinements leave the subnormalization at $\alpha^L$. That exponential growth in
$L$ is intrinsic to the product rule and is *not* an artifact: even a matrix
with spectral radius $\rho(A)<1$, so that $\sup_L\norm{A^L}\le C<\infty$, may have
$\norm{A}>1$ and hence require $\alpha\ge\norm{A}>1$; encoding $A^L$ this way carries
subnormalization $\alpha^L$ even though $\norm{A^L}$ stays bounded. Reconciling these
two facts---bounded powers, blowing-up subnormalization---is exactly the difficulty
that dedicated linear-ODE algorithms (that chapter) are built to resolve.

**Example (Two-term product).**

Let $A,B$ be $1$-sparse with $\norm{A}_{\max},\norm{B}_{\max}\le1$, block-encoded by
$U_A\in\mathrm{BE}_{1,1}(A)$ and $U_B\in\mathrm{BE}_{1,1}(B)$ as in
that result. Using *two* flag qubits (registers $a$ and $b$), the
circuit $U_{AB}=(U_A\otimes\id)(\id\otimes U_B)$ is a $(1,2)$-block-encoding of
$AB$: on flag $\ket0\ket0$ it returns $AB\ket\psi$ with the two flag qubits back in
$\ket0$. If instead we tried to reuse a single flag qubit
($U_AU_B$ on the same ancilla), the $\ket1$ branch produced by $U_B$ would be fed
into $U_A$ and re-enter the $\ket0$ block, corrupting the answer---the failure of
that remark. The extra ancilla is precisely the bookkeeping that keeps the
two encodings from interfering.

## Hybrid LCU and importance sampling

The circuit $W$ of that theorem realizes $T=\sum_j c_j U_j$
*coherently*: all $K$ branches exist simultaneously in superposition, and a
single run applies the whole combination. Often, though, we do not want the state
$T\ket{\psi}$ itself but only an *expectation value*
$u\dg O u$ of some observable $O$ in the (unnormalized) output
$u=T\ket{u_0}=\sum_j c_j U_j\ket{u_0}$. In that case a *hybrid* strategy---run
one term at a time on the quantum computer, combine classically---becomes available
and can trade circuit depth for repetitions.

Expanding the quadratic form,
$$u\dg O u
  =\sum_{j,j'}c_j c_{j'}\,\bra{u_0}U_j\dg\,O\,U_{j'}\ket{u_0}.$$
Each summand $\bra{u_0}U_j\dg O U_{j'}\ket{u_0}$ is an off-diagonal matrix element
that a quantum computer can estimate with a *Hadamard test*: prepare
$\ket{u_0}$, apply the controlled pair $U_j\dg O U_{j'}$ (or, for a single unitary
$O$, the controlled $U_{j'}$ and $U_j$), and read the ancilla. No superposition over
$j$ is required---only two of the $U$'s appear per run---so the circuit is as shallow
as a single term, at the price of estimating an $\bigo(K^2)$-term sum.

**Proposition (Importance-sampling estimator for $u\dg Ou$).**

Let $c_j\ge0$, $\norm{c}_1=\sum_j c_j$, and define the probability distribution
$p_j=c_j/\norm{c}_1$ on $[K]$. Draw pairs $(j,j')$ independently with
$\Prob[(j,j')]=p_j p_{j'}$, and for each pair form the (Hadamard-test) unbiased
estimate $\widehat{h}_{jj'}$ of $h_{jj'}\defeq\bra{u_0}U_j\dg O U_{j'}\ket{u_0}$.
Then
$$\widehat{\mathcal E}\defeq\norm{c}_1^2\,\widehat{h}_{jj'}
  \qquad\text{is an unbiased estimator of } u\dg Ou,$$
and averaging over $R$ independent samples gives variance
$\Var[\tfrac1R\sum\widehat{\mathcal E}]=\bigo(\norm{c}_1^4\,\norm{O}^2/R)$. Hence
$\bigo(\norm{c}_1^4\norm{O}^2/\varepsilon^2)$ samples suffice to estimate $u\dg Ou$
to additive error $\varepsilon$.

*Proof.*

Unbiasedness: with $(j,j')\sim p_jp_{j'}$ and $\widehat h_{jj'}$ conditionally
unbiased for $h_{jj'}$,
$$\E[\widehat{\mathcal E}]
  =\norm{c}_1^2\sum_{j,j'}p_j p_{j'}\,h_{jj'}
  =\norm{c}_1^2\sum_{j,j'}\frac{c_j}{\norm{c}_1}\frac{c_{j'}}{\norm{c}_1}h_{jj'}
  =\sum_{j,j'}c_j c_{j'}h_{jj'}
  =u\dg Ou,$$
by the equation above. For the variance, each $\abs{h_{jj'}}\le\norm{O}$
(since $\norm{U_j\ket{u_0}}=1$), so a single estimate obeys
$\abs{\widehat{\mathcal E}}\le\norm{c}_1^2\norm{O}$ (up to the $\bigo(1)$ Hadamard-test
variance), giving $\Var[\widehat{\mathcal E}]=\bigo(\norm{c}_1^4\norm{O}^2)$; the
$R$-sample average divides this by $R$. Setting the standard deviation to
$\varepsilon$ yields $R=\bigo(\norm{c}_1^4\norm{O}^2/\varepsilon^2)$.

**Remark (Coherent vs. hybrid: the trade-off).**

The two implementations occupy opposite ends of a depth--repetition trade-off.

-   *Coherent LCU* (that theorem) uses one deep circuit containing
    all $K$ terms via $\mathrm{SEL}$, with an ancilla register and (if the state
    is wanted) $\bigo(\norm{c}_1/\norm{u})$ amplifications; it produces the
    quantum state $u/\norm{u}$ and preserves coherence for further processing.

-   *Hybrid importance sampling* (that result) uses many shallow
    circuits with at most two terms each and *no* multiplexed $\mathrm{SEL}$,
    but pays a worse dependence on the $\ell^1$-norm---$\norm{c}_1^4$ in the
    sample count versus $\norm{c}_1$ in the coherent repeat count---and only ever
    yields a scalar $u\dg Ou$, not the state.

The hybrid route is attractive on near-term hardware, where circuit depth and clean
ancillas are scarce, and when the final quantity of interest is a single expectation
value; the coherent route wins asymptotically and whenever the output state must be
fed into a downstream quantum subroutine.

## Cost: the $\ell^1$ overhead

Every construction above is governed by one scalar: the $\ell^1$-norm of the
combination coefficients. Collecting the accounting:

-   *Addition* of block encodings with subnormalizations $\alpha_j$ produces
    subnormalization $\gamma=\sum_j\alpha_j$ (that result); for a
    weighted sum $\sum_j c_j A_j$ with $\norm{A_j}\le1$ the subnormalization is
    $\norm{c}_1$. The number of times the coherent circuit must be run to see the
    flag is $\bigo(\gamma/\norm{T\ket{\psi}})$
    (that remark), reduced to $\bigo(\gamma/\norm{T})$ by oblivious
    amplitude amplification when $T$ is proportional to an isometry
    (that remark).

-   *Multiplication* of $L$ block encodings produces subnormalization
    $\prod_j\alpha_j$ (that result, that remark). For
    $\alpha_j\equiv\alpha$ this is $\alpha^L$, an exponential in the number of
    factors.

-   *Hybrid sampling* costs $\bigo(\norm{c}_1^4\norm{O}^2/\varepsilon^2)$
    shallow runs (that result).

The moral is that $\norm{c}_1$ is to LCU what $\alpha$ was to a single block encoding
(that chapter): a subnormalization that shrinks the success probability
quadratically and multiplies every downstream query count linearly (or, for
products, exponentially in the number of factors). Minimizing it is therefore the
principal design problem.

**Example (The $\ell^1$-norm depends on the decomposition).**

A given $T$ can be written as a linear combination of unitaries in many ways, and the
resulting $\norm{c}_1$ varies. Consider the diagonal matrix
$T=\diag(1,1,\dots,1,-1)$ on $n$ qubits (a single sign flip on the last basis
state). One decomposition writes $T=\id-2\proj{N-1}$; since $\proj{N-1}$ is not
unitary, expand it into Paulis, $\proj{N-1}=\prod_{k}\tfrac{\id-\Zgate_k}{2}$, whose
expansion has $2^n$ Pauli terms each with coefficient $2^{-n}$, giving
$\norm{c}_1=\id\text{-term}+2\cdot2^{-n}\cdot 2^n=1+2=3$ after cancellation---but the
naive term-by-term bound is $\norm{c}_1=1+2\sum_k\ldots$, exponentially worse if one
is careless. A far better decomposition recognizes $T$ itself as a single
*reflection* (it is unitary and Hermitian), so $T=1\cdot T$ with $\norm{c}_1=1$
and no LCU overhead at all. The lesson: before invoking LCU, check whether the target
is already (a scalar times) a unitary, and otherwise seek the decomposition with the
smallest $\ell^1$-norm. This is why Hamiltonians are grouped into as few Pauli terms
as possible, and why product formulas (that chapter) sometimes beat LCU for
simulation.

**Remark (Where LCU sits in the toolbox).**

LCU and single-matrix block encoding (that chapter) together generate the full
arithmetic of block encodings: sums, weighted sums, and products of encodable
matrices are all encodable, with the bookkeeping rules of this lecture. What LCU does
*not* directly give is a general *function* $f(A)$ beyond polynomials
expressible as short LCUs (e.g. truncated Chebyshev or Fourier series). For that we
need qubitization and quantum signal processing (that chapter) and the quantum
singular value transformation (that chapter), which realize essentially arbitrary
bounded polynomials of $A$ with subnormalization independent of the polynomial
degree---often dramatically better than the $\norm{c}_1$ of the corresponding LCU. We
will return to compare the two approaches once QSVT is in hand.

## Exercises {#exercises .unnumbered}

**Exercise.**

*Deriving the LCU block.* Let $T=\sum_{j\in[K]}c_j U_j$ with $c_j\ge0$,
$K=2^a$, and let $W=(\mathrm{PREP}\dg\otimes\id)\,\mathrm{SEL}\,(\mathrm{PREP}
\otimes\id)$ as in that theorem. Reproduce the three-line computation
the equation above--the equation above to show
$(\bra{0^a}\otimes\id)W(\ket{0^a}\otimes\id)=T/\norm{c}_1$. Then show that if the
input flag is $\ket{k}\ne\ket{0^a}$ instead of $\ket{0^a}$, the surviving
$\bra{0^a}$-block is $\frac{1}{\norm{c}_1}\sum_j\sqrt{c_j}\,
\overline{(\mathrm{PREP})_{jk}}\,\sqrt{c_j}\,U_j$ in general---i.e. only the
$\ket{0^a}$ column of $\mathrm{PREP}$ matters for the encoded operator.

**Exercise.**

*Optimality of $\norm{c}_1$.* Using that remark, show that for any
factorization $c_j=\beta_j\gamma_j$ the asymmetric circuit block-encodes $T$ with
subnormalization $\norm{\beta}_2\norm{\gamma}_2$, and prove via Cauchy--Schwarz that
this is minimized, at value $\norm{c}_1$, by $\beta_j=\gamma_j=\sqrt{c_j}$. Conclude
that among all [prepare]{.smallcaps}--[select]{.smallcaps}--[unprepare]{.smallcaps} circuits for a
fixed $\{U_j\}$, the symmetric one has the smallest subnormalization.

**Exercise.**

*LCU of three Paulis.* Consider $T=\Xgate+\Ygate+2\Zgate$ on one qubit.
Determine $\norm{c}_1$, the number of control qubits $a$, and write down explicitly
the prepare state $\mathrm{PREP}\ket{0^a}$ and the select oracle
$\mathrm{SEL}=\sum_j\proj{j}\otimes U_j$ (you may use a $2$-qubit control register
with one label unused, and set the coefficient of the unused label to $0$). State the
resulting block-encoding parameters $(\alpha,a)$ and compute $\norm{T}$ to check that
$\alpha=\norm{c}_1$ overshoots $\norm{T}$ here.

**Exercise.**

*Block-encoding a weighted sum.* Let $U_A\in\mathrm{BE}_{\alpha_A,a}(A)$ and
$U_B\in\mathrm{BE}_{\alpha_B,a}(B)$ share a flag register. Following
that result, construct a circuit that block-encodes $C=A+B$ and identify
its subnormalization and ancilla count. Then do the same for a general
$\alpha_A\ne\alpha_B$, writing down the required $\mathrm{PREP}$ amplitudes
$\propto(\sqrt{\alpha_A},\sqrt{\alpha_B})$, and confirm the subnormalization is
$\alpha_A+\alpha_B$.

**Exercise.**

*Why naive multiplication fails.* Let $U_A\in\mathrm{BE}_{1,1}(A)$ and
$U_B\in\mathrm{BE}_{1,1}(B)$ share a *single* flag qubit. Write $U_B\ket0\ket
\psi=\ket0 B\ket\psi+\ket1\ket{\chi}$ and $U_A(\ket0 B\ket\psi+\ket1\ket\chi)$, and
show by an explicit $2\times2$ example ($A=B=\tfrac12\Xgate$, say, so
$\norm{A}=\norm{B}=\tfrac12$) that
$(\bra0\otimes\id)U_AU_B(\ket0\otimes\id)\ne AB$ in general because of the
$\ket1\ket\chi$ cross term. Then verify that placing $U_A,U_B$ on *disjoint*
flag qubits (that result) restores $(\bra{00}\otimes\id)U_{AB}
(\ket{00}\otimes\id)=AB$.

**Exercise.**

*Subnormalization of a power.* Let $A$ be block-encoded by
$U_A\in\mathrm{BE}_{\alpha,a}(A)$ with $\alpha=\norm{A}$. Using
that result iteratively (or the compression gadget of
that remark), give a block encoding of $A^L$ and its subnormalization.
Exhibit a $2\times2$ matrix $A$ with $\rho(A)<1$ but $\norm{A}>1$ (hint: a
nonnormal matrix such as $\bigl(\begin{smallmatrix}0&2\\0&0\end{smallmatrix}\bigr)$
scaled so that $\rho=0$, or a Jordan-type block), and explain why the
subnormalization $\alpha^L$ grows without bound even though $\norm{A^L}$ does not.

**Exercise.**

*Cost of the hybrid estimator.* In the setting of that result, suppose
$K=100$ terms with all $c_j=1$ (so $\norm{c}_1=100$) and a bounded observable
$\norm{O}\le1$. How many shallow Hadamard-test runs $R$ are needed to estimate
$u\dg Ou$ to additive error $\varepsilon=10^{-2}$? Compare with the coherent-LCU
repeat count $\bigo(\norm{c}_1/\norm{u})$ (assume $\norm{u}=\Theta(\norm{c}_1)$) and
comment on which implementation you would choose (i) on a shallow near-term device
and (ii) if the state $u/\norm{u}$ is needed for a subsequent quantum subroutine.

## Further reading {#further-reading .unnumbered}

-   Lin Lin, *qasc notes*, Sec. 7.3 (linear combination of unitaries).

-   Lin Lin, *live notes*, Ch. 9 (LCU lemma, block encodings of matrix
    additions and multiplications, and worked examples including the highly
    oscillatory integral and Fourier/eigenvalue transformations).

-   Dong An, *Quantum Numerical Linear Algebra* tutorial talk (LCU as
    [prepare]{.smallcaps}--[select]{.smallcaps}--[unprepare]{.smallcaps}, matrix and vector
    addition, matrix multiplication by duplicate ancillas and the compression
    gadget, and the hybrid importance-sampling estimator).

-   G. H. Low and I. L. Chuang, *Hamiltonian simulation by qubitization*,
    Quantum **3**, 163 (2019), and A. M. Childs and N. Wiebe,
    *Hamiltonian simulation using linear combinations of unitary
    operations*, Quantum Inf. Comput. **12**, 901 (2012), for the original
    LCU and oblivious amplitude amplification arguments.